In [160]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
import time

In [7]:
website=requests.get("https://books.toscrape.com/catalogue/page-1.html").text

In [159]:
from re import findall
soup = BeautifulSoup(website,'lxml')

# print(soup.find('p',class_='star-rating')['class'][1])
soup.find('p',class_='price_color').text[1:]#

soup.find('h3').find('a')['title']#find_all() returns a list (needs [0], [1]...)
#find() returns a single Tag (needs ['attr'] to pull data out of it, never [0]).

soup.find('p',class_='instock availability').text.strip()

#found book link now can fetch description
ur=requests.get("https://books.toscrape.com/catalogue/"+soup.find('h3').find('a')['href']).text
bosoup = BeautifulSoup(ur,'lxml')
bosoup.find('meta',attrs={'name': 'description'})['content'].strip()

#find category
bosoup.find('ul',class_="breadcrumb").find_all('a')[2].text #output:mystery

"Iris Johansen's beloved forensic sculptor Eve Duncan is back and now the stakes are higher than ever. Dramatic changes are on the horizon for Eve and Joe Quinn and their relationship may never be the same. Faced with the task of protecting Cara Delaney, a young girl with ruthless enemies who want to see her dead, Eve takes her away to the remote Scottish Highlands where th Iris Johansen's beloved forensic sculptor Eve Duncan is back and now the stakes are higher than ever. Dramatic changes are on the horizon for Eve and Joe Quinn and their relationship may never be the same. Faced with the task of protecting Cara Delaney, a young girl with ruthless enemies who want to see her dead, Eve takes her away to the remote Scottish Highlands where they join Jane MacGuire in search of a hidden treasure. But nowhere is far enough away to protect Cara from danger. With enemies closing in from all sides, Hide Away is a high-octane thriller that fans will not want to miss. ...more"

In [173]:
title=[]
rating=[]
price=[]
stock=[]
category=[]
description=[]

for i in range(1,51):

  website=requests.get("https://books.toscrape.com/catalogue/page-{}.html".format(i)).text
  soup = BeautifulSoup(website,'lxml')
  books=soup.find_all('article',class_='product_pod')

  for j in books:

    try:
      title.append(j.find('h3').find('a')['title'])
    except:
      title.append(np.nan)

    try:
      rating.append(j.find('p',class_='star-rating')['class'][1])
    except:
      rating.append(np.nan)

    try:
      price.append(j.find('p',class_='price_color').text[1:].strip())
    except:
      price.append(np.nan)

    try:
      stock.append(j.find('p',class_='instock availability').text.strip())
    except:
      stock.append(np.nan)

    title_tag=j.find('h3').find('a')['href']

    try:
      bookurl="https://books.toscrape.com/catalogue/"+title_tag
      book_url=requests.get(bookurl).text
      book_soup=BeautifulSoup(book_url,'lxml')

    except:
      book_soup=None

    time.sleep(0.1)

    if book_soup:
      try:
        description.append(book_soup.find('meta',attrs={'name': 'description'})['content'].strip())
      except:
        description.append(np.nan)
      try:
        crumbs = book_soup.find('ul', class_='breadcrumb').find_all('li')
        category.append(crumbs[2].find('a').text)
      except:
        category.append(np.nan)

    else:
      description.append(np.nan)
      category.append(np.nan)




In [174]:

data={
    'title':title,
    'category':category,
    'rating':rating,
    'description':description,
    'stock':stock,
    'price':price

}
book1=pd.DataFrame(data)

In [175]:
rating_map = {'One':1,'Two':2,'Three':3,'Four':4,'Five':5}
book1['rating'] = book1['rating'].map(rating_map)

In [176]:
book1

,title,category,rating,description,stock,price
0,A Light in the Attic,Poetry,3,It's hard to imagine a world without A Light i...,In stock,£51.77
1,Tipping the Velvet,Historical Fiction,1,"""Erotic and absorbing...Written with starling ...",In stock,£53.74
2,Soumission,Fiction,1,"Dans une France assez proche de la nÃ´tre, un ...",In stock,£50.10
3,Sharp Objects,Mystery,4,"WICKED above her hipbone, GIRL across her hear...",In stock,£47.82
4,Sapiens: A Brief History of Humankind,History,5,From a renowned historian comes a groundbreaki...,In stock,£54.23
...,...,...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,Classics,1,,In stock,£55.53
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",Sequential Art,4,High school student Kei Nagai is struck dead i...,In stock,£57.06
997,A Spy's Devotion (The Regency Spies of London #1),Historical Fiction,5,"In Englandâs Regency era, manners and elegan...",In stock,£16.97
998,1st to Die (Women's Murder Club #1),Mystery,1,"James Patterson, bestselling author of the Ale...",In stock,£53.98


In [177]:
book1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        1000 non-null   object
 1   category     1000 non-null   object
 2   rating       1000 non-null   int64 
 3   description  1000 non-null   object
 4   stock        1000 non-null   object
 5   price        1000 non-null   object
dtypes: int64(1), object(5)
memory usage: 47.0+ KB


In [178]:
book1.to_csv('books_dataset.csv', index=False)

In [179]:
book1['category'].value_counts()

,count
category,
Default,152
Nonfiction,110
Sequential Art,75
Add a comment,67
Fiction,65
Young Adult,54
Fantasy,48
Romance,35
Mystery,32
